<a href="https://colab.research.google.com/github/Katyayini-Sharma/Celebal-Internship-2026/blob/main/week_8_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [3]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

import re
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("agent")

# Only digits, decimal points, whitespace, parentheses, and + - * / % are allowed
# through to eval(). This blocks things like "Calculate import os" or "Calculate __import__('os')"
# from ever reaching the calculator tool.
SAFE_EXPRESSION_PATTERN = re.compile(r"^[\d\s\.\+\-\*/%()]+$")

# Connector words that often sit right after "keywords" ("keywords from ...",
# "keywords for ...") and aren't part of the actual content to analyze.
KEYWORD_CONNECTOR_PATTERN = re.compile(r"^(from|for|in|of|about)\s+", flags=re.IGNORECASE)


def agent(query: str):
    """Route a query to the right tool and return a structured JSON-style response."""

    # --- Guard: reject non-string input up front ---
    if not isinstance(query, str) or not query.strip():
        logger.warning("Rejected non-string or empty input: %r", query)
        return {"type": "error", "result": "Query must be a non-empty string"}

    try:
        query_lower = query.lower()

        # --- Route 1: math queries -> Calculator Tool ---
        # (checked first, so a query mentioning both "calculate" and "keywords"
        # is deterministically treated as a calculation request)
        if re.search(r"\bcalculate\b", query_lower):
            match = re.search(r"calculate\s*:?\s*(.*)", query, flags=re.IGNORECASE)
            expression = match.group(1).strip() if match else ""

            if not expression:
                logger.info("Calculate route: no expression found in %r", query)
                return {"type": "error", "result": "No expression found after 'calculate'"}

            if not SAFE_EXPRESSION_PATTERN.match(expression):
                logger.warning("Calculate route: unsafe expression blocked: %r", expression)
                return {"type": "error", "result": f"Invalid characters in expression: '{expression}'"}

            calc_result = calculator(expression)
            if calc_result == "Error in calculation":
                logger.info("Calculate route: calculator failed on %r", expression)
                return {"type": "error", "result": calc_result}

            logger.info("Calculate route: %r -> %s", expression, calc_result)
            return {"type": "calculation", "result": calc_result}

        # --- Route 2: keyword extraction -> Keyword Tool ---
        elif re.search(r"\bkeywords\b", query_lower):
            # Only analyze the text AFTER the trigger word "keywords", so words like
            # "extract" and "keywords" itself never leak into the extracted results.
            parts = re.split(r"\bkeywords\b", query, maxsplit=1, flags=re.IGNORECASE)
            content = parts[1].strip() if len(parts) > 1 else ""
            content = KEYWORD_CONNECTOR_PATTERN.sub("", content)  # drop a leading "from"/"for"/etc.

            if not content:
                logger.info("Keywords route: no content found after trigger word in %r", query)
                return {"type": "error", "result": "No text provided for keyword extraction"}

            keywords = extract_keywords(content)
            if not keywords:
                logger.info("Keywords route: nothing extracted from %r", content)
                return {"type": "error", "result": "No keywords could be extracted"}

            logger.info("Keywords route: %r -> %s", content, keywords)
            return {"type": "keywords", "result": keywords}

        # --- Route 3: everything else -> general fallback response ---
        else:
            logger.info("General route: %r", query)
            return {
                "type": "general",
                "result": f"I received your query: '{query}'. Could you clarify if you'd like a calculation or keyword extraction?"
            }

    # --- Safety net: malformed input / unexpected failures -> explicit error label ---
    except Exception as e:
        logger.error("Unexpected error handling %r: %s", query, e)
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [4]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['intelligence', 'transforming', 'industries', 'artificial']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I received your query: 'What is machine learning?'. Could you clarify if you'd like a calculation or keyword extraction?"}
--------------------------------------------------


In [5]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): Calculate 2+3
Response: {'type': 'calculation', 'result': '5'}
Enter query (type 'exit' to stop): extract keywords from my name is katyayini and i study in btech final year
Response: {'type': 'keywords', 'result': ['btech', 'final', 'katyayini', 'study']}
Enter query (type 'exit' to stop): hello world
Response: {'type': 'general', 'result': "I received your query: 'hello world'. Could you clarify if you'd like a calculation or keyword extraction?"}
Enter query (type 'exit' to stop): exit
